In [0]:
%pip install kagglehub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import kagglehub
import shutil
import os
from pyspark.sql.functions import current_timestamp, lit

volume_root = "/Volumes/dbacademy/default/ecommerce_project"
raw_olist_path = f"{volume_root}/raw/olist"
raw_rees46_path = f"{volume_root}/raw/rees46"
bronze_base = f"{volume_root}/bronze"

olist_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
os.makedirs(raw_olist_path, exist_ok=True)
for file in os.listdir(olist_path):
    shutil.copy(os.path.join(olist_path, file), raw_olist_path)
print("Olist files staged:", os.listdir(raw_olist_path))

rees46_path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")
os.makedirs(raw_rees46_path, exist_ok=True)
shutil.copy(os.path.join(rees46_path, "2019-Oct.csv"), raw_rees46_path)
print("REES46 file staged:", os.listdir(raw_rees46_path))

olist_files = os.listdir(raw_olist_path)
for filename in olist_files:
    if filename == "olist_order_reviews_dataset.csv":
        continue
    table_name = filename.replace("_dataset.csv", "").replace(".csv", "")
    df = spark.read.csv(f"{raw_olist_path}/{filename}", header=True, inferSchema=True)
    df = df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_file", lit(filename))
    df.write.format("delta").mode("overwrite").save(f"{bronze_base}/{table_name}")
    print(f"{table_name}: {df.count()} rows written")

df_reviews = spark.read.csv(
    f"{raw_olist_path}/olist_order_reviews_dataset.csv",
    header=True, inferSchema=True, multiLine=True, quote='"', escape='"'
)
df_reviews = df_reviews.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_file", lit("olist_order_reviews_dataset.csv"))
df_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_base}/olist_order_reviews")
print("olist_order_reviews:", df_reviews.count(), "rows written")

from pyspark.sql.functions import col
df_rees46 = spark.read.csv(f"{raw_rees46_path}/2019-Oct.csv", header=True, inferSchema=True)
df_rees46_sample = df_rees46.filter((col("event_time") >= "2019-10-01") & (col("event_time") < "2019-10-08"))
df_rees46_sample = df_rees46_sample.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_file", lit("2019-Oct.csv (sampled: Oct 1-7)"))
df_rees46_sample.write.format("delta").mode("overwrite").save(f"{bronze_base}/rees46_events")
print("rees46_events:", df_rees46_sample.count(), "rows written")

print("=== Bronze ingestion complete ===")

100%|██████████| 42.6M/42.6M [00:00<00:00, 45.1MB/s]

Extracting files...


Olist files staged: ['olist_customers_dataset.csv', 'olist_products_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


100%|██████████| 4.29G/4.29G [00:38<00:00, 121MB/s]

Extracting files...


REES46 file staged: ['2019-Oct.csv']
olist_customers: 99441 rows written
olist_products: 32951 rows written
olist_order_payments: 103886 rows written
olist_geolocation: 1000163 rows written
olist_orders: 99441 rows written
olist_order_items: 112650 rows written
olist_sellers: 3095 rows written
product_category_name_translation: 71 rows written
olist_order_reviews: 99224 rows written
rees46_events: 8829315 rows written
=== Bronze ingestion complete ===


In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count as spark_count, when, coalesce, lit,
    to_timestamp, avg, first, split, expr
)
from pyspark.sql import Row

bronze_base = "/Volumes/dbacademy/default/ecommerce_project/bronze"
silver_base = "/Volumes/dbacademy/default/ecommerce_project/silver"

# --- Orders ---
df_orders = spark.read.format("delta").load(f"{bronze_base}/olist_orders")
df_orders_silver = df_orders \
    .withColumn("is_delivered", when(col("order_status") == "delivered", True).otherwise(False)) \
    .drop("ingestion_timestamp", "source_file")
df_orders_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/orders")
print("orders:", df_orders_silver.count())

# --- Customers ---
df_customers_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_customers")
df_customers_silver = df_customers_bronze.drop("ingestion_timestamp", "source_file")
df_customers_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/customers")
print("customers:", df_customers_silver.count())

# --- Order Items ---
df_items_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_order_items")
df_items_silver = df_items_bronze.drop("ingestion_timestamp", "source_file")
df_items_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/order_items")
print("order_items:", df_items_silver.count())

# --- Order Payments ---
df_payments_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_order_payments")
df_payments_silver = df_payments_bronze \
    .filter(col("payment_type") != "not_defined") \
    .withColumn("payment_installments",
                when(col("payment_installments") == 0, 1).otherwise(col("payment_installments"))) \
    .drop("ingestion_timestamp", "source_file")
df_payments_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/order_payments")
print("order_payments:", df_payments_silver.count())

# --- Order Reviews ---
df_reviews_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_order_reviews")
df_reviews_silver = df_reviews_bronze \
    .withColumn("review_score", col("review_score").cast("integer")) \
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date"))) \
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp"))) \
    .drop("ingestion_timestamp", "source_file")
df_reviews_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/order_reviews")
print("order_reviews:", df_reviews_silver.count())

# --- Products ---
df_products_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_products")
medians = df_products_bronze.approxQuantile(
    ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    [0.5], 0.01
)
median_weight, median_length, median_height, median_width = [m[0] for m in medians]
df_products_silver = df_products_bronze \
    .withColumn("product_category_name", coalesce(col("product_category_name"), lit("unknown"))) \
    .withColumn("product_name_lenght", coalesce(col("product_name_lenght"), lit(0))) \
    .withColumn("product_description_lenght", coalesce(col("product_description_lenght"), lit(0))) \
    .withColumn("product_photos_qty", coalesce(col("product_photos_qty"), lit(0))) \
    .withColumn("product_weight_g", coalesce(col("product_weight_g"), lit(median_weight))) \
    .withColumn("product_length_cm", coalesce(col("product_length_cm"), lit(median_length))) \
    .withColumn("product_height_cm", coalesce(col("product_height_cm"), lit(median_height))) \
    .withColumn("product_width_cm", coalesce(col("product_width_cm"), lit(median_width))) \
    .drop("ingestion_timestamp", "source_file")
df_products_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/products")
print("products:", df_products_silver.count())

# --- Sellers ---
df_sellers_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_sellers")
df_sellers_silver = df_sellers_bronze.drop("ingestion_timestamp", "source_file")
df_sellers_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/sellers")
print("sellers:", df_sellers_silver.count())

# --- Geolocation (deduplicated by zip code) ---
df_geo_bronze = spark.read.format("delta").load(f"{bronze_base}/olist_geolocation")
df_geo_silver = df_geo_bronze.groupBy("geolocation_zip_code_prefix").agg(
    avg("geolocation_lat").alias("geolocation_lat"),
    avg("geolocation_lng").alias("geolocation_lng"),
    first("geolocation_city").alias("geolocation_city"),
    first("geolocation_state").alias("geolocation_state")
)
df_geo_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/geolocation")
print("geolocation:", df_geo_silver.count())

# --- Category Translation ---
df_cat_bronze = spark.read.format("delta").load(f"{bronze_base}/product_category_name_translation")
df_cat_silver = df_cat_bronze.drop("ingestion_timestamp", "source_file")
df_cat_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/category_translation")
print("category_translation:", df_cat_silver.count())

# --- REES46 Events (category hierarchy parsing + null handling) ---
df_rees46_bronze = spark.read.format("delta").load(f"{bronze_base}/rees46_events")
df_rees46_silver = df_rees46_bronze \
    .withColumn("category_code", coalesce(col("category_code"), lit("uncategorized"))) \
    .withColumn("brand", coalesce(col("brand"), lit("unknown"))) \
    .withColumn("category_split", split(col("category_code"), "\\.")) \
    .withColumn("category_level_1", expr("try_element_at(category_split, 1)")) \
    .withColumn("category_level_2", expr("try_element_at(category_split, 2)")) \
    .withColumn("category_level_3", expr("try_element_at(category_split, 3)")) \
    .drop("category_split", "ingestion_timestamp", "source_file")
df_rees46_silver.write.format("delta").mode("overwrite").save(f"{silver_base}/rees46_events")
print("rees46_events:", df_rees46_silver.count())

# --- Category Mapping (Olist -> REES46 bucket) ---
category_mapping = {
    "agro_industry_and_commerce": "no_match", "air_conditioning": "appliances", "art": "no_match",
    "arts_and_craftmanship": "no_match", "audio": "electronics", "auto": "auto", "baby": "kids",
    "bed_bath_table": "furniture", "books_general_interest": "no_match", "books_imported": "no_match",
    "books_technical": "no_match", "cds_dvds_musicals": "electronics", "christmas_supplies": "no_match",
    "cine_photo": "electronics", "computers": "computers", "computers_accessories": "computers",
    "consoles_games": "electronics", "construction_tools_construction": "construction",
    "construction_tools_lights": "construction", "construction_tools_safety": "construction",
    "cool_stuff": "no_match", "costruction_tools_garden": "construction", "costruction_tools_tools": "construction",
    "diapers_and_hygiene": "kids", "drinks": "no_match", "dvds_blu_ray": "electronics",
    "electronics": "electronics", "fashio_female_clothing": "apparel", "fashion_bags_accessories": "accessories",
    "fashion_childrens_clothes": "apparel", "fashion_male_clothing": "apparel", "fashion_shoes": "apparel",
    "fashion_sport": "sport", "fashion_underwear_beach": "apparel", "fixed_telephony": "electronics",
    "flowers": "country_yard", "food": "no_match", "food_drink": "no_match", "furniture_bedroom": "furniture",
    "furniture_decor": "furniture", "furniture_living_room": "furniture",
    "furniture_mattress_and_upholstery": "furniture", "garden_tools": "country_yard",
    "health_beauty": "medicine", "home_appliances": "appliances", "home_appliances_2": "appliances",
    "home_comfort_2": "appliances", "home_confort": "appliances", "home_construction": "construction",
    "housewares": "appliances", "industry_commerce_and_business": "no_match",
    "kitchen_dining_laundry_garden_furniture": "furniture", "la_cuisine": "appliances",
    "luggage_accessories": "accessories", "market_place": "no_match", "music": "electronics",
    "musical_instruments": "no_match", "office_furniture": "furniture", "party_supplies": "no_match",
    "perfumery": "medicine", "pet_shop": "no_match", "security_and_services": "no_match",
    "signaling_and_security": "no_match", "small_appliances": "appliances",
    "small_appliances_home_oven_and_coffee": "appliances", "sports_leisure": "sport",
    "stationery": "stationery", "tablets_printing_image": "computers", "telephony": "electronics",
    "toys": "kids", "watches_gifts": "accessories",
}
mapping_rows = [Row(product_category_name_english=k, rees46_category_bucket=v) for k, v in category_mapping.items()]
df_category_mapping = spark.createDataFrame(mapping_rows)
df_category_mapping.write.format("delta").mode("overwrite").save(f"{silver_base}/category_mapping")
print("category_mapping:", df_category_mapping.count())

print("=== Silver transformation complete ===")

orders: 99441
customers: 99441
order_items: 112650
order_payments: 103883
order_reviews: 99224
products: 32951
sellers: 3095
geolocation: 19015
category_translation: 71
rees46_events: 8829315
category_mapping: 71
=== Silver transformation complete ===
